In [ ]:
import Pkg;
Pkg.activate("../")
Pkg.instantiate()

# Numerical oracle for the constants

In this section we do some numerical computations to narrow down the set of parameters. Later on, we will use self-validated numerical methods (Interval Arithmetic) to certify the 
numerical values we computed now. We do this since numerical computations are inexpensive, while self validated methods may be more time-consuming.

In [ ]:
using Plots

In [ ]:
Pkg.status()

We fix the truncation size for the Galerkin approximation.

In [ ]:
K = 256
N = 2*K+1

We start by defining the large perturbation of the doubling map
$$
T(z) = i z^2 \cdot exp\left[\left(\frac{1}{2}-\frac{\pi}{16}\right)\left(z-\frac{1}{z}\right)\right].
$$

In [ ]:
T(z) = im*z^2*exp((1/2-π/16)*(z-1/z))

Let $A_{r} = \{z \mid e^{-2\pi r}\leq |z| \leq e^{2\pi r}\}$.

We are interested in finding $\eta$, $\rho$ such that the closure $A_{\rho}$ is contained in $B_{\mu}(A_{\eta})$.
We are interested in maximizing $\rho-\eta$, since it is the constant appearing in the main error term of our functional analytic treatment, i.e.:
$$
||Lf-L_Kf||_{\ell^1}\leq \left(1+2\frac{e^{-2 \pi |\rho-\alpha|}}{1-e^{-2 \pi |\rho-\alpha|}}\right)\left(e^{-2\pi K\alpha}+e^{-2\pi K(\alpha-\eta)}\right)||f||_{\infty, \alpha}.	
$$

For $\rho>1$ fix
$$
\alpha_o(\rho):=\min_{\theta \in [0,1]}|B(\rho e^{2\pi i \theta})|
$$
where $_0$ stays for outer and 
$$
OR(\rho) := \frac{1}{2\pi}\left(\log(\alpha_o(\rho))-\log(\rho)\right).
$$
We would like to maximize the function $OR$.

In [ ]:
α_o(ρ) = minimum(abs.(B.([ρ * exp(im * 2 * π * θ) for θ in 0:0.001:1])))
function OR(ρ)   
    return (log(α_o(ρ))-log(ρ))/2π
end

Similarly, we would like to treat the image inside the circle; for $\rho>1$ we define 
$$
\alpha_i(\rho) :=\min_{\theta \in [0,1]}\frac{1}{\left|B\left(\frac{e^{2\pi i \theta}}{\rho}\right)\right|}
$$
and 
$$
IR(\rho) := \frac{1}{2\pi}\left(\log(\alpha_i(\rho))-\log(\rho)\right).
$$


In [ ]:
α_i(ρ) = minimum(1.0 ./abs.(B.([exp(im * 2 * π * θ) / ρ for θ in 0:0.001:1])))
    
function IR(ρ) 
    return (log(α_i(ρ))-log(ρ))/2*π
end 

We define now 
$$
LR(\rho) = \min\{IR(\rho),OR(\rho)\}
$$
and maximise it.

In [ ]:
LR(ρ) = min(IR(ρ), OR(ρ))

In [ ]:
plot(LR, 1, 100)

In [ ]:
η_rad = 1:0.1:10

bestrad, indexrad = findmax([LR(η) for η in η_rad])

η = η_rad[indexrad]

We move now to the strips

In [ ]:
η_s = log(η)/2π
ρ_s = log(min(α_i(η), α_o(η)))/2π
η_s, ρ_s, ρ_s-η_s    

We want to find now an $\alpha$ that minimizes the right hand side of
$$
||Lf-L_Kf||_{\ell^1}\leq \left(1+2\frac{e^{-2 \pi |\rho-\alpha|}}{1-e^{-2 \pi |\rho-\alpha|}}\right)\left(e^{-2\pi K\alpha}+e^{-2\pi K(\alpha-\eta)}\right)||f||_{\infty, \alpha}	
$$


In [ ]:
function rhs(α, K; η, ρ)
    Dρα = ρ-α
    Dαη = α-η
    coeff_1 = 1+2*(exp(-2*π*(Dρα)))/(1-(exp(-2*π*(Dρα))))
    coeff_2 = exp(-2*π*K*α)+exp(-2*π*K*(Dαη))
    return coeff_1*coeff_2
end

In [ ]:
plot(α -> rhs(α, K; η = η_s, ρ = ρ_s), η_s, ρ_s)

In [ ]:
α_arr = LinRange(η_s, ρ_s, 10000)
val_min, idx = findmin(map(α -> rhs(α, K; η = η_s, ρ = ρ_s), α_arr))
α_s = α_arr[idx]
val_min, α_s

We compute now the right hand side, supposing $|\mu|>1/2$
$$
||f||_{\infty, \alpha} \leq \left(\frac{1}{|\mu|}\right)^{\frac{\alpha}{\alpha-\eta}} \left( 1+2\frac{e^{-2 \pi |\rho-\alpha|}}{1-e^{-2 \pi |\rho-\alpha|}}\right)^{\frac{\alpha}{\alpha-\eta}} ||f||_{\ell^1}.
$$


In [ ]:
function weak_strong(μ; η, α, ρ)
    s = α/(α-η)
    coeff_1 = (1/abs(μ))^s
    coeff_2 = 1+2*(exp(-2*π*(ρ-α)))/(1-(exp(-2*π*(ρ-α))))^s
    return coeff_1*coeff_2
end

In [ ]:
bws = weak_strong(0.5; η = η_s, α = α_s, ρ = ρ_s)

Clearly, the number above is ugly, but the fact that the projection error is small balances out. The `sqrt(N)` in the formula is the constant that relates
the $||.||_{\ell^1}$ norm and the $||.||_{\ell^2}$ norm. 

In [ ]:
(bws*val_min)*sqrt(N)

# Certifying the constants

For the specific values of $\alpha$, $\rho$ and $\eta$ computed above, we will certify the value of the constants.

In [ ]:
using IntervalArithmetic

In [ ]:
Ir = sqrt(interval(2))*17/32 
Iϕ = interval(π) / 8

max_r = 10.0
Iμ = Ir * exp(im * Iϕ)
IB(z; μ=Iμ) = (z * (μ - z)) / (1 - μ' * z)

In [ ]:
N = 16*1048576
Iα_o(ρ) = minimum(abs.(IB.([ρ * exp(im * 2 * interval(π) * interval(i, i+1)/N) for i in 0:N-1])))

In [ ]:
Iα_i(ρ) = minimum(1.0 ./abs.(IB.([exp(im * 2 * interval(π) * interval(i, i+1)/N) / ρ for i in 0:N-1])))

In [ ]:
α_o(ρ_s), α_i(ρ_s)

In [ ]:
ibound = Iα_i(ρ_s)

In [ ]:
obound = Iα_o(ρ_s)

In [ ]:
using RigorousInvariantMeasures, BallArithmetic

In [ ]:

FourierBasis = RigorousInvariantMeasures.FourierAdjoint(K, 65536)

In [ ]:
S(x) = 0.5 + atan((sin(2 * pi * x) - r * sin(ϕ)) / (cos(2 * pi * x) - r * cos(ϕ))) / pi

In [ ]:
plot(S, 0, 1)

In [ ]:
savefig("Blashke.png")

In [ ]:
P = DiscretizedOperator(FourierBasis, S)

In [ ]:
import IntervalArithmetic
midI = IntervalArithmetic.mid
radI = IntervalArithmetic.radius

In [ ]:
midP = midI.(real.(P.L)) + im * midI.(imag.(P.L))

In [ ]:
radP = sqrt.(radI.(real.(P.L))^2 + radI.(imag.(P.L))^2)

In [ ]:
BallP = BallMatrix(midP, radP)

In [ ]:
using Pseudospectra

In [ ]:
spectralportrait(midP)

In [ ]:
savefig("PseudospectraGalerkinBlashke.png")

In [ ]:
using LinearAlgebra
abs.(diag(schur(midP).T))

With this discretization size we are not going to separate 
things that are separated by less than $0.0025$.
By inspecting the eigenvalues of the Schur matrix, we see that it may be difficult to separate the eigenvalues with norm $0.522...$, that corresponde to a double eigenvalue.
Therefore we resolve to enclose a bigger circle than $0.5$.  